<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Equality Constraints and Feasible Decisions

**Choose clock settings that make one recorded reading exactly correct.**

Lecture 03-1 found an unrestricted initial correction and hourly rate. We keep the same observations and squared-error objective. We now add an exact calibration requirement and see how it restricts the decisions.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Keep the score and add one requirement

The recorded errors $d_i$ are $-2,-1,-1,0$ min at times $t_i=0,1,2,3$ h, for $i=1,\ldots,4$.
Choose an initial correction $q$ in minutes and a rate correction $r$ in min/h. The remaining errors and their score are

$$
y_i=d_i+q+rt_i,\qquad
\phi(q,r)=\sum_{i=1}^{4}y_i^2.
$$

At one hour, the recorded error is $-1$ min. Making that reading correct requires
$q+(1\,\mathrm h)r=1\,\mathrm{min}$.
Using numerical values in these stated units, write the **equality residual** as

$$
h(q,r)=q+r-1=0.
$$

A residual is the left side after moving the requirement to zero. A decision is **feasible** when it satisfies that requirement.
The previous unrestricted answer, $(q,r)=(1.9,-0.6)$, gives $h=0.3$, so it is no longer acceptable.

The next calculation keeps the observations, requirement, remaining errors, score, and feasibility check separate. The small tolerance only handles floating-point roundoff; it does not change the mathematical requirement.


In [ ]:
import numpy as np

# Fixed observations and their elapsed times
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h

# Calibration requirement: correction at 1 h must equal 1 min
CALIBRATION_TIME = 1.0        # h
REQUIRED_CORRECTION = 1.0     # min
CHECK_TOLERANCE = 1e-9        # numerical comparison setting


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    remaining_errors = simulate_clock(decision)
    correction, rate = decision
    equality_residual = correction + CALIBRATION_TIME * rate - REQUIRED_CORRECTION
    gradient = 2 * np.array([np.sum(remaining_errors),
                             np.dot(OBSERVATION_TIMES, remaining_errors)])
    return {
        "decision": decision,
        "response": remaining_errors,
        "objective": float(np.sum(remaining_errors ** 2)),
        "gradient": gradient,
        "equality_residual": float(equality_residual),
        "feasible": bool(abs(equality_residual) <= CHECK_TOLERANCE),
    }


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Compare the old answer with two decisions that satisfy the new requirement. Reject an infeasible decision before comparing its score with feasible decisions.


In [ ]:
for decision in ([1.9, -0.6], [1.0, 0.0], [1.5, -0.5]):
    result = evaluate_clock(decision)
    if not result["feasible"]:
        print(f"{decision}: reject; equality residual = {result['equality_residual']:.2f} min")
        continue
    print(f"{decision}: feasible | errors = {result['response']} min"
          f" | score = {result['objective']:.2f} min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Change settings without leaving the equality

The equality $q+r=1$ means that $r=1-q$. Increasing the numerical value of $q$ by 0.1 requires decreasing the numerical value of $r$ by 0.1. The units remain min and min/h, respectively.

For example, $(1,0)$ and $(1.1,-0.1)$ are both feasible. The setting $(1.1,0)$ is not. An acceptable change must preserve the relationship between the two decisions.

Substituting $r=1-q$ turns this particular constrained problem into a one-variable problem. This substitution is convenient for one simple equality.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Check every feasible decision

The equality lets us replace $r$ by $1-q$. Substitution and completing the square give

$$
\phi(q,1-q)=6q^2-18q+14
=6(q-1.5)^2+0.5.
$$

The square cannot be negative. Its value is zero only at $q=1.5$, which forces $r=-0.5$.
This proves the unique minimum **among decisions satisfying the equality**, with score 0.5 min².
The unrestricted score of 0.2 min² was lower, but its decision violates the new requirement.

For example, increasing $q$ from 1.5 to 1.6 min requires changing $r$ from $-0.5$ to $-0.6$ min/h to keep calibration. The score becomes $6(1.6-1.5)^2+0.5=0.56$ min², above the minimum of 0.5 min².

Compare a few points on the allowed line. This numerical comparison illustrates the identity above; the identity proves the result for every real $q$ on that line.


In [ ]:
for correction in (1.0, 1.4, 1.5, 1.6, 2.0):
    rate = REQUIRED_CORRECTION - correction
    result = evaluate_clock([correction, rate])
    print(f"q = {correction:.1f}, r = {rate:.1f} | feasible = {result['feasible']}"
          f" | score = {result['objective']:.2f} min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · See the allowed line and its minimum

The left panel shows equal-score contours and the line of feasible settings. The gray cross is the unrestricted answer, which misses the line.
The right panel shows the score only along the allowed line. Compare the orange feasible candidate with the teal constrained minimum.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-2_constrained_optimization_lagrange_multipliers/assets/clock_equality_constraint.png" alt="The equality q plus r equals 1 restricts the two clock settings to a line. The unrestricted minimum lies off that line; the constrained minimum is q equal to 1.5 and r equal to minus 0.5, with score 0.5 square minutes." width="1000" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The plotting function uses the same evaluation records as the calculations.


In [ ]:
import matplotlib.pyplot as plt

BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
GRAY = "#909090"


def show_equality_constraint():
    figure, axes = plt.subplots(1, 2, figsize=(12, 5.2))
    q_values = np.linspace(0.0, 2.5, 181)
    r_values = np.linspace(-1.0, 1.0, 181)
    q_mesh, r_mesh = np.meshgrid(q_values, r_values)
    scores = sum((error + q_mesh + time * r_mesh) ** 2
                 for error, time in zip(OBSERVED_ERRORS, OBSERVATION_TIMES))
    axes[0].contour(q_mesh, r_mesh, scores, levels=[0.25, 0.5, 1, 2, 4, 8, 16],
                    colors=BLUE, alpha=0.45)
    axes[0].plot(q_values, 1 - q_values, color=TEAL, label="Allowed line: q + r = 1")
    axes[0].scatter(1.9, -0.6, marker="x", s=85, color=GRAY, linewidths=2,
                    label="Unrestricted answer: infeasible", zorder=5)
    axes[0].scatter(1.0, 0.0, color=ORANGE, s=60, label="Feasible candidate", zorder=5)
    axes[0].scatter(1.5, -0.5, marker="D", color=TEAL, s=70,
                    label="Constrained minimum", zorder=6)
    axes[0].set(xlabel="Initial correction q (min)", ylabel="Rate correction r (min/h)",
                xlim=(0, 2.5), ylim=(-1, 1), title="The equality restricts the allowed settings")
    axes[0].legend(loc="upper right", fontsize=8)
    line_scores = [evaluate_clock([q, 1 - q])["objective"] for q in q_values]
    axes[1].plot(q_values, line_scores, color=BLUE, label="Score with r = 1 - q")
    for q, color, marker, label in [(1.0, ORANGE, "o", "Feasible candidate"),
                                    (1.5, TEAL, "D", "Constrained minimum")]:
        result = evaluate_clock([q, 1 - q])
        axes[1].scatter(q, result["objective"], color=color, marker=marker, s=70, label=label)
    axes[1].set(xlabel="Initial correction q (min)", ylabel="Squared-error score (min²)",
                title="The lowest feasible score is 0.5 min²")
    axes[1].legend(fontsize=8)
    for axis in axes:
        axis.grid(alpha=0.25)
    figure.text(0.5, 0.025, "The plotted ranges limit the view; the equality is the only requirement.",
                ha="center", fontsize=9)
    figure.tight_layout(rect=(0, 0.065, 1, 1))
    return figure


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Draw the two views of the same constrained clock problem.


In [ ]:
equality_figure = show_equality_constraint()
plt.show()
